# 🏥 Asistente Virtual Clínico (Chatbot de Triaje) - IA Avanzada

**Autor:** Erick Guardo, Einer plaza
**Asignatura:** Inteligencia Artificial Avanzada
**Profesor:** Carlos Carrascal Avendaño

Este cuaderno (*notebook*) de Google Colab implementa la arquitectura **RAG (Retrieval-Augmented Generation)** para un chatbot clínico. 
Se divide en las siguientes fases operativas:
1. **Instalación de Dependencias**: `transformers`, `torch`, `pymongo` (para bases de datos).
2. **Modelo de Recuperación (Retriever)**: Uso de *ClinicalBERT* para entender síntomas médicos.
3. **Base de Datos (Simulación MongoDB)**: Para la verificación de médicos disponibles y horarios.
4. **Pruebas de Equidad (Fairness)**: Auditoría algorítmica para evitar sesgos.

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS NECESARIAS
!pip install transformers torch scikit-learn pymongo -q
print("Librerías instaladas correctamente.")

In [ ]:
# 2. IMPORTACIÓN DE LIBRERÍAS
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json

print("Librerías importadas. Entorno listo.")

## Arquitectura RAG (Recuperador Clínico)
Aquí cargamos **ClinicalBERT**, un modelo entrenado específicamente en historias clínicas (notas de UCI), lo que le permite entender lenguaje médico mejor que un modelo genérico.

In [ ]:
class ClinicalRetriever:
    def __init__(self):
        print("Descargando/Cargando ClinicalBERT en Colab...")
        self.tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        
        # Base de datos vectorial (Protocolos Médicos Reales)
        self.knowledge_base = [
            "Protocolo Infarto: Dolor opresivo en el pecho, irradiado al brazo izquierdo, sudoración. Acción: Llamar a urgencias (Código Rojo).",
            "Protocolo Resfriado: Tos leve, congestión nasal, sin fiebre alta. Acción: Descanso, cita rutinaria (Código Verde).",
            "Protocolo Ansiedad: Palpitaciones, sensación de ahogo sin causa física. Acción: Respiración, psicología (Código Amarillo)."
        ]
        self.kb_embeddings = self._compute_embeddings(self.knowledge_base)

    def _compute_embeddings(self, texts):
        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
        with torch.no_grad():
            outputs = self.model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        return F.normalize(embeddings, p=2, dim=1)

    def retrieve(self, query):
        query_emb = self._compute_embeddings([query])
        sims = cosine_similarity(query_emb.numpy(), self.kb_embeddings.numpy())[0]
        best_idx = np.argmax(sims)
        
        # Mitigación de Alucinaciones
        if sims[best_idx] < 0.3:
            return "Protocolo no encontrado. Derivar a humano."
        return self.knowledge_base[best_idx]

retriever = ClinicalRetriever()
print("Modelo Retriever cargado y listo en memoria.")

## Simulación de Conexión a Base de Datos Externa (Agendamiento de Citas)
En un entorno de producción, aquí nos conectaríamos a MongoDB (usando `pymongo`). En este cuaderno, simulamos la base de datos.

In [ ]:
class DatabaseMock:
    def __init__(self):
        self.medicos = {
            "Cardiologia": [{"nombre": "Dr. Perez", "disponible": False}], # Ocupado
            "Medicina General": [{"nombre": "Dra. Gomez", "disponible": True}] # Disponible
        }
    
    def buscar_medico_disponible(self, especialidad):
        if especialidad in self.medicos:
            for medico in self.medicos[especialidad]:
                if medico["disponible"]:
                    return medico["nombre"]
        return None

db = DatabaseMock()

## Lógica del Generador y Orquestador del Chatbot
Conectando la API (WhatsApp entrante simulada) -> Recuperador Clínico -> Base de Datos -> Respuesta.

In [ ]:
def procesar_mensaje_whatsapp(mensaje_paciente):
    print(f"\n[WhatsApp] Paciente dice: '{mensaje_paciente}'")
    
    # 1. IA analiza los síntomas
    contexto_medico = retriever.retrieve(mensaje_paciente)
    print(f"[Backend IA] Protocolo Recuperado: {contexto_medico}")
    
    # 2. Generación de Respuesta y Lógica de Negocio
    if "Código Rojo" in contexto_medico:
        print("[Respuesta Chatbot]: ⚠️ ALERTA: Acuda a URGENCIAS inmediatamente. Riesgo vital.")
        # Consulta a DB si hay cardiologo por si acaso
        medico = db.buscar_medico_disponible("Cardiologia")
        if not medico:
            print("[Respuesta Chatbot]: (Aviso Interno: Todos los cardiólogos están ocupados, derive a hospital externo).")
            
    elif "Código Verde" in contexto_medico:
        print("[Respuesta Chatbot]: Sus síntomas son leves.")
        medico = db.buscar_medico_disponible("Medicina General")
        if medico:
            print(f"[Respuesta Chatbot]: ¿Desea agendar cita rutinaria con {medico}? (Sí/No)")
        else:
            print("[Respuesta Chatbot]: No hay médicos disponibles hoy. Intente mañana.")
            
    else:
        print("[Respuesta Chatbot]: Por favor, reescriba sus síntomas.")


## Interfaz Interactiva de Prueba (Simulador WhatsApp)
Esta celda abre un menú interactivo para que puedas chatear en tiempo real (escribe el número de opción).

In [ ]:
import time

print("*"*40)
print("🤖 Bienvenido al Chatbot Clínico")
print("*"*40)

while True:
    print("\nOpciones disponibles:")
    print("1️⃣ Agendar Cita")
    print("2️⃣ Evaluación Médica (Triaje de IA)")
    print("3️⃣ Salir")
    
    opcion = input("Escribe el número de tu opción (1, 2 o 3): ")
    
    if opcion == '1':
        print("\n[Chatbot]: Entendido. Redirigiendo a la base de datos de citas...")
        time.sleep(1)
        medico = db.buscar_medico_disponible("Medicina General")
        if medico:
            print(f"[Chatbot]: Tenemos a {medico} disponible hoy. Para confirmar, llame al 01-8000.")
        else:
            print("[Chatbot]: Lo sentimos, no hay médicos disponibles en este momento.")
            
    elif opcion == '2':
        print("\n[Chatbot]: Has elegido el Triaje con Inteligencia Artificial.")
        sintomas = input("Por favor, escribe tus síntomas médicos aquí: ")
        print("\n⏳ Analizando con Redes Neuronales Profundas (ClinicalBERT)...")
        time.sleep(1.5)
        procesar_mensaje_whatsapp(sintomas)
        
    elif opcion == '3':
        print("\n[Chatbot]: Gracias por usar el sistema. ¡Cuídate mucho! 👋")
        break
        
    else:
        print("\n[Chatbot]: ⚠️ Opción no válida, por favor intenta de nuevo.")
